
# Masterclass Notebook: Interpretabilidad vs Explicabilidad de Modelos
**Caso de estudio:** Titanic Survival Prediction  
**Autor:** Josef Rodriguez  
**Objetivo:** comparar un modelo interpretable con modelos tipo *black box* y aplicar métodos modernos de explicación.

---

## Estructura de la clase

1. Problema Titanic  
2. Logistic Regression → modelo interpretable  
3. Random Forest → modelo black box  
4. XGBoost / LightGBM / CatBoost → modelos boosting  
5. Feature Importance  
6. Permutation Importance  
7. Partial Dependence / ICE  
8. SHAP global  
9. SHAP individual  
10. Conclusiones industriales

---

## Pregunta de negocio

> Queremos predecir si una persona sobrevivió o no al Titanic a partir de variables observables como sexo, edad, clase, tarifa, puerto de embarque y estructura familiar.

Este problema es ideal porque:
- es fácil de entender por los alumnos,
- tiene variables muy interpretables,
- permite comparar **modelos simples** vs **modelos complejos**,
- funciona muy bien con técnicas de explicabilidad modernas.



## 1. Diferencia conceptual

### Interpretabilidad
Un modelo es **interpretable** cuando podemos entender directamente cómo toma decisiones.

Ejemplos:
- Regresión lineal
- Regresión logística
- Árboles pequeños
- Modelos scorecard / WoE

### Explicabilidad
La **explicabilidad** aparece cuando el modelo no es transparente por sí mismo, pero usamos herramientas externas para entenderlo.

Ejemplos:
- SHAP
- Permutation Importance
- Partial Dependence Plots (PDP)
- ICE plots
- LIME

### Idea clave de la clase
- **Logistic Regression** representa la parte interpretable.
- **Random Forest / XGBoost / LightGBM / CatBoost** representan modelos de mayor complejidad.
- Luego usamos métodos de explicación para abrir la “caja negra”.


In [ ]:

# =============================
# 0. Instalación opcional
# =============================
# Ejecuta esta celda solo si te faltan librerías.
# En muchos entornos no será necesario.

# %pip install -q pandas numpy matplotlib scikit-learn seaborn shap xgboost lightgbm catboost


In [ ]:

# =============================
# 1. Imports
# =============================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Opcionales
HAS_XGB = True
HAS_LGBM = True
HAS_CAT = True
HAS_SHAP = True

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier
except Exception:
    HAS_CAT = False

try:
    import shap
except Exception:
    HAS_SHAP = False

print("XGBoost disponible:", HAS_XGB)
print("LightGBM disponible:", HAS_LGBM)
print("CatBoost disponible:", HAS_CAT)
print("SHAP disponible:", HAS_SHAP)


In [ ]:

# =============================
# 2. Carga robusta del dataset Titanic
# =============================
def load_titanic():
    '''
    Intenta cargar Titanic desde seaborn y, si falla, usa OpenML.
    Devuelve un DataFrame estandarizado.
    '''
    try:
        import seaborn as sns
        df = sns.load_dataset("titanic")
        source = "seaborn"
    except Exception:
        from sklearn.datasets import fetch_openml
        titanic = fetch_openml(name="titanic", version=1, as_frame=True)
        df = titanic.frame.copy()
        source = "openml"

    # Estandarización mínima de columnas para que el notebook sea consistente
    rename_map = {
        "survived": "survived",
        "pclass": "pclass",
        "sex": "sex",
        "age": "age",
        "sibsp": "sibsp",
        "parch": "parch",
        "fare": "fare",
        "embarked": "embarked",
        "class": "class",
        "who": "who",
        "adult_male": "adult_male",
        "deck": "deck",
        "embark_town": "embark_town",
        "alone": "alone",
    }
    existing = [c for c in rename_map if c in df.columns]
    df = df[existing].copy()

    # Algunas fuentes pueden traer el target como string/categoría
    if df["survived"].dtype == "object":
        df["survived"] = df["survived"].astype(int)

    return df, source

df, source = load_titanic()
print("Fuente:", source)
print("Shape:", df.shape)
df.head()



## 2.1 Variables del problema

- **survived**: variable objetivo (1 = sobrevivió, 0 = no sobrevivió)
- **pclass**: clase del ticket
- **sex**: sexo
- **age**: edad
- **sibsp**: número de hermanos/cónyuges a bordo
- **parch**: número de padres/hijos a bordo
- **fare**: tarifa pagada
- **embarked**: puerto de embarque

Además, aprovecharemos otras variables derivadas o descriptivas si están disponibles.


In [ ]:

# =============================
# 3. Revisión rápida del dataset
# =============================
df.info()


In [ ]:

df.isnull().mean().sort_values(ascending=False)


In [ ]:

# Distribución del target
target_dist = df["survived"].value_counts(normalize=True).sort_index()
print(target_dist)

ax = target_dist.plot(kind="bar")
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("survived")
ax.set_ylabel("proporción")
plt.show()


In [ ]:

# Relación simple y muy intuitiva: supervivencia por sexo
survival_by_sex = df.groupby("sex")["survived"].mean().sort_values(ascending=False)
survival_by_sex


In [ ]:

ax = survival_by_sex.plot(kind="bar")
ax.set_title("Supervivencia promedio por sexo")
ax.set_ylabel("probabilidad de sobrevivir")
plt.show()


In [ ]:

# Relación con clase
survival_by_class = df.groupby("pclass")["survived"].mean().sort_index()
survival_by_class


In [ ]:

ax = survival_by_class.plot(kind="bar")
ax.set_title("Supervivencia promedio por clase")
ax.set_ylabel("probabilidad de sobrevivir")
plt.show()



## 3.1 Lectura de negocio preliminar

Antes de modelar ya vemos señales útiles:
- las mujeres tendieron a sobrevivir más,
- los pasajeros de clase alta tendieron a sobrevivir más,
- la edad y la estructura familiar probablemente influyen.

Esto es muy valioso pedagógicamente porque luego podremos comprobar si los modelos capturan estos patrones.


In [ ]:

# =============================
# 4. Feature engineering ligero
# =============================
work_df = df.copy()

work_df["family_size"] = work_df["sibsp"].fillna(0) + work_df["parch"].fillna(0) + 1
work_df["is_alone_flag"] = (work_df["family_size"] == 1).astype(int)
work_df["fare_per_person"] = work_df["fare"] / work_df["family_size"].replace(0, 1)

# Variables finales
target = "survived"
features = [c for c in work_df.columns if c != target]

work_df.head()



## 4.1 Comentario didáctico

Aquí agregamos algunas variables sencillas:
- **family_size**
- **is_alone_flag**
- **fare_per_person**

Este paso es importante porque muestra que incluso en una clase de explicabilidad, el *feature engineering* sigue siendo parte fundamental del trabajo del Data Scientist.


In [ ]:

# =============================
# 5. Train / test split
# =============================
X = work_df[features].copy()
y = work_df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)


In [ ]:

# =============================
# 6. Preprocesamiento
# =============================
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)



## 5. Modelos a comparar

### Modelo interpretable
- **Logistic Regression**

### Modelos black-box / semi black-box
- **Random Forest**
- **XGBoost**
- **LightGBM**
- **CatBoost**

La idea no es solo comparar métricas, sino ver qué herramientas necesitamos para explicar cada familia de modelos.


In [ ]:

# =============================
# 7. Definición de modelos
# =============================
models = {}

models["LogisticRegression"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

models["RandomForest"] = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42
    ))
])

if HAS_XGB:
    models["XGBoost"] = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=42
        ))
    ])

if HAS_LGBM:
    models["LightGBM"] = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbose=-1
        ))
    ])

if HAS_CAT:
    models["CatBoost"] = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", CatBoostClassifier(
            iterations=300,
            depth=5,
            learning_rate=0.05,
            verbose=False,
            random_state=42
        ))
    ])

list(models.keys())


In [ ]:

# =============================
# 8. Entrenamiento y evaluación
# =============================
results = []
fitted_models = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    proba_train = pipe.predict_proba(X_train)[:, 1]
    proba_test = pipe.predict_proba(X_test)[:, 1]
    pred_test = (proba_test >= 0.5).astype(int)

    results.append({
        "model": name,
        "train_auc": roc_auc_score(y_train, proba_train),
        "test_auc": roc_auc_score(y_test, proba_test),
        "test_accuracy": accuracy_score(y_test, pred_test)
    })

results_df = pd.DataFrame(results).sort_values("test_auc", ascending=False).reset_index(drop=True)
results_df


In [ ]:

# Visualización de métricas
ax = results_df.set_index("model")[["train_auc", "test_auc"]].plot(kind="bar")
ax.set_title("Comparación de AUC train vs test")
ax.set_ylabel("AUC")
plt.xticks(rotation=45)
plt.show()



## 6. Primera lectura analítica

Con esta tabla puedes discutir:
- si hay diferencias materiales entre modelos,
- si hay señales de sobreajuste,
- si el modelo más complejo realmente mejora,
- cuánto sacrificamos en interpretabilidad al pasar de una logística a un boosting.

En clase, aquí vale mucho la pena preguntar:

> ¿Vale la pena usar un modelo más complejo si la mejora predictiva es marginal?


In [ ]:

# Elegimos el mejor modelo por test_auc para la parte de explicabilidad avanzada
best_model_name = results_df.loc[0, "model"]
best_pipe = fitted_models[best_model_name]

print("Mejor modelo según test_auc:", best_model_name)



# 7. Interpretabilidad directa: Logistic Regression

La logística es un ejemplo clásico de **modelo interpretable**, porque sus coeficientes tienen una lectura relativamente directa sobre el log-odds de la probabilidad objetivo.

## Fórmula

\[
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k
\]

### Lectura
- Si \( \beta_j > 0 \), la variable empuja hacia mayor probabilidad de sobrevivir.
- Si \( \beta_j < 0 \), la variable empuja hacia menor probabilidad de sobrevivir.
- Si exponenciamos \( \beta_j \), obtenemos un **odds ratio** aproximado.


In [ ]:

# =============================
# 9. Coeficientes de la logística
# =============================
log_pipe = fitted_models["LogisticRegression"]

feature_names = log_pipe.named_steps["prep"].get_feature_names_out()
coefs = log_pipe.named_steps["model"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
    "odds_ratio": np.exp(coefs)
}).sort_values("coef")

coef_df.head(15)


In [ ]:

coef_df.tail(15)


In [ ]:

# Top efectos positivos y negativos
top_neg = coef_df.head(10).sort_values("coef")
top_pos = coef_df.tail(10).sort_values("coef")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_neg["feature"], top_neg["coef"])
axes[0].set_title("Logística: variables que reducen supervivencia")
axes[0].set_xlabel("coeficiente")

axes[1].barh(top_pos["feature"], top_pos["coef"])
axes[1].set_title("Logística: variables que aumentan supervivencia")
axes[1].set_xlabel("coeficiente")

plt.tight_layout()
plt.show()



## 7.1 Cómo explicar esto en clase

Ejemplos típicos:
- un coeficiente positivo para `sex_female` indica que ser mujer aumenta la probabilidad de supervivencia,
- un coeficiente negativo en ciertas categorías de clase o variables asociadas a vulnerabilidad reduce la probabilidad,
- el signo importa más que la magnitud bruta cuando las escalas son diferentes, aunque aquí escalamos las numéricas para facilitar lectura.

**Punto docente clave:** la logística permite una narrativa causal-operativa más clara, aunque no necesariamente es el modelo más preciso.



# 8. Modelos black-box: importancia de variables

En modelos como Random Forest, XGBoost, LightGBM o CatBoost, ya no podemos “leer” la decisión con la misma facilidad que en una logística.

Por eso usamos herramientas de explicación.

La primera es la **Feature Importance** interna del modelo.


In [ ]:

# =============================
# 10. Feature importance interna
# =============================
def get_model_feature_importance(pipe):
    model = pipe.named_steps["model"]
    feature_names = pipe.named_steps["prep"].get_feature_names_out()

    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
        return pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False)
    else:
        return None

for candidate in ["RandomForest", "XGBoost", "LightGBM", "CatBoost"]:
    if candidate in fitted_models:
        fi_df = get_model_feature_importance(fitted_models[candidate])
        if fi_df is not None:
            print(f"\n=== {candidate} ===")
            display(fi_df.head(10))


In [ ]:

# Gráfico del mejor modelo si tiene feature_importances_
best_fi = get_model_feature_importance(best_pipe)
if best_fi is not None:
    top = best_fi.head(15).sort_values("importance")
    plt.figure(figsize=(8, 6))
    plt.barh(top["feature"], top["importance"])
    plt.title(f"Feature Importance interna - {best_model_name}")
    plt.xlabel("importance")
    plt.show()
else:
    print("El mejor modelo no expone feature_importances_.")



## 8.1 Advertencia metodológica

La **feature importance interna** es útil, pero no siempre es suficiente.

Problemas comunes:
- puede sesgarse por variables con muchas particiones,
- depende de la lógica interna del algoritmo,
- no siempre refleja impacto causal ni dirección,
- no te explica bien casos individuales.

Por eso conviene complementar con métodos más robustos.



# 9. Permutation Importance

Este método responde a una pregunta muy útil:

> ¿Qué tanto empeora el modelo si destruimos la información de una variable?

La lógica es:
1. medimos el desempeño del modelo normal,
2. permutamos una variable,
3. volvemos a medir,
4. si el desempeño cae mucho, esa variable era importante.

**Ventaja:** es más agnóstico al modelo.


In [ ]:

# =============================
# 11. Permutation Importance
# =============================
perm = permutation_importance(
    best_pipe,
    X_test,
    y_test,
    n_repeats=15,
    random_state=42,
    scoring="roc_auc"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(15)


In [ ]:

top_perm = perm_df.head(15).sort_values("importance_mean")
plt.figure(figsize=(8, 6))
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.title(f"Permutation Importance - {best_model_name}")
plt.xlabel("caída promedio de AUC al permutar")
plt.show()



## 9.1 Lectura docente

Este gráfico suele ser muy bueno en clase porque es intuitivo:
- si al romper una variable el modelo empeora mucho, esa variable importa,
- si no cambia casi nada, la variable aporta poco.

Además, es excelente para comparar contra la importancia interna y discutir por qué no siempre coinciden.



# 10. Partial Dependence (PDP) e ICE

SHAP no es el único método.

## PDP
Muestra el efecto promedio de una variable sobre la predicción del modelo.

## ICE
Muestra trayectorias individuales, es decir, cómo cambia la predicción para observaciones concretas al variar una variable.

**Uso docente:** ayudan a responder:
- ¿la relación es lineal?
- ¿hay umbrales?
- ¿hay saturación?
- ¿hay heterogeneidad entre individuos?


In [ ]:

# =============================
# 12. PDP / ICE
# =============================
candidate_features = []
for f in ["age", "fare", "family_size", "pclass"]:
    if f in X_test.columns:
        candidate_features.append(f)

candidate_features


In [ ]:

# Para pipelines, PartialDependenceDisplay puede trabajar directamente sobre X original.
for feat in candidate_features[:3]:
    fig, ax = plt.subplots(figsize=(7, 4))
    PartialDependenceDisplay.from_estimator(
        best_pipe,
        X_test,
        [feat],
        kind="both",   # both = PDP + ICE
        ax=ax
    )
    ax.set_title(f"PDP + ICE para {feat} - {best_model_name}")
    plt.show()



## 10.1 Cómo leer PDP / ICE

- La línea promedio (PDP) resume el efecto medio.
- Las líneas individuales (ICE) muestran si ese efecto es homogéneo o no.
- Si las curvas individuales divergen mucho, la relación depende del contexto de cada observación.

Esto permite introducir una idea muy poderosa:
> una variable puede ser importante, pero su efecto no necesariamente es lineal ni igual para todos.



# 11. Otras técnicas además de SHAP

En esta clase conviene mencionarlas explícitamente:

## A. Coeficientes / Odds Ratios
Ideales para modelos lineales o scorecards.

## B. Feature Importance
Útil, rápida, pero limitada.

## C. Permutation Importance
Más agnóstica y comparativa.

## D. PDP / ICE
Muy buenas para relaciones no lineales y umbrales.

## E. LIME
Explica observaciones individuales perturbando alrededor del punto.

## F. Surrogate Models
Entrenas un modelo simple para aproximar a uno complejo y ganar interpretación.

## G. Rule Extraction
Extraes reglas de decisión comprensibles desde modelos complejos o árboles.

### Idea docente clave
SHAP es muy poderoso, pero **no es la única herramienta**.  
Un Data Scientist serio no se casa con una sola técnica: triangula evidencia.



# 12. SHAP: teoría breve

SHAP viene de los valores de Shapley de teoría de juegos cooperativos.

La predicción se descompone como:

\[
f(x) = \phi_0 + \sum_{j=1}^{p} \phi_j
\]

Donde:
- \( \phi_0 \): valor base del modelo,
- \( \phi_j \): contribución de cada variable a la predicción.

**Ventajas de SHAP**
- da explicaciones locales y globales,
- tiene base teórica sólida,
- permite ver magnitud y dirección del efecto,
- sirve muy bien para tree-based models.

**Desventaja**
- puede ser más costoso computacionalmente,
- requiere cuidado en interpretación cuando hay alta correlación entre variables.


In [ ]:

# =============================
# 13. Preparación de matrices transformadas para SHAP
# =============================
X_train_trans = best_pipe.named_steps["prep"].transform(X_train)
X_test_trans = best_pipe.named_steps["prep"].transform(X_test)
best_estimator = best_pipe.named_steps["model"]
best_feature_names = best_pipe.named_steps["prep"].get_feature_names_out()

# Algunas librerías devuelven sparse matrix. Convertimos a denso solo si hace falta.
if hasattr(X_train_trans, "toarray"):
    X_train_shap = X_train_trans.toarray()
else:
    X_train_shap = X_train_trans

if hasattr(X_test_trans, "toarray"):
    X_test_shap = X_test_trans.toarray()
else:
    X_test_shap = X_test_trans

X_train_shap_df = pd.DataFrame(X_train_shap, columns=best_feature_names)
X_test_shap_df = pd.DataFrame(X_test_shap, columns=best_feature_names)

X_train_shap_df.head()


In [ ]:

# =============================
# 14. SHAP explainer
# =============================
if HAS_SHAP:
    try:
        explainer = shap.Explainer(best_estimator, X_train_shap_df)
        shap_values = explainer(X_test_shap_df)
        print("Explainer construido correctamente para:", best_model_name)
    except Exception as e:
        print("No se pudo construir shap.Explainer genérico:", e)
        shap_values = None
else:
    shap_values = None
    print("SHAP no disponible en este entorno.")



# 13. SHAP global

La vista global responde preguntas como:
- ¿qué variables dominan el modelo?
- ¿qué dirección tienen?
- ¿cuánta dispersión muestran?
- ¿hay variables con efecto muy no lineal?

En un **summary plot**:
- el eje X muestra el impacto SHAP,
- el color representa valores altos/bajos de la variable,
- cada punto es una observación.


In [ ]:

# =============================
# 15. SHAP summary plot
# =============================
if shap_values is not None:
    shap.plots.beeswarm(shap_values, max_display=15)
else:
    print("No hay shap_values disponibles.")



## 13.1 Cómo interpretar el beeswarm

Ejemplo de lectura:
- puntos a la derecha empujan hacia mayor probabilidad de sobrevivir,
- puntos a la izquierda empujan hacia menor probabilidad,
- si una variable tiene gran dispersión horizontal, su impacto cambia mucho entre casos,
- el color ayuda a entender si valores altos o bajos de la variable impulsan el resultado.

Este gráfico suele ser el punto más “wow” de la clase.



# 14. SHAP individual

La explicación local responde:

> ¿Por qué este pasajero en particular fue clasificado como sobreviviente o no sobreviviente?

Esto es crucial en negocio real:
- rechazo de crédito,
- sospecha de fraude,
- recomendación médica,
- priorización de clientes.


In [ ]:

# =============================
# 16. Selección de observación individual
# =============================
row_idx = 0
print("Índice local en test:", row_idx)
display(X_test.iloc[[row_idx]])
print("Target real:", int(y_test.iloc[row_idx]))
print("Probabilidad predicha:", round(best_pipe.predict_proba(X_test.iloc[[row_idx]])[:, 1][0], 4))


In [ ]:

# Waterfall plot local
if shap_values is not None:
    shap.plots.waterfall(shap_values[row_idx], max_display=12)
else:
    print("No hay shap_values disponibles.")



## 14.1 Cómo leer el waterfall

- Partimos de un valor base.
- Cada variable empuja la predicción hacia arriba o hacia abajo.
- El valor final explica la probabilidad del caso observado.

Esto permite contar historias de negocio muy concretas:

> “Este pasajero tuvo alta probabilidad de sobrevivir porque era mujer, viajaba en cierta clase y tenía determinados atributos; sin embargo, otros factores restaron algo de probabilidad”.



# 15. Comparación metodológica final

## Logistic Regression
**Fortaleza:** interpretación directa  
**Debilidad:** puede perder capacidad predictiva ante relaciones complejas

## Random Forest
**Fortaleza:** robusto y flexible  
**Debilidad:** menos transparente

## XGBoost / LightGBM / CatBoost
**Fortaleza:** alto desempeño tabular  
**Debilidad:** más complejos de explicar sin herramientas extra

## Métodos de explicación revisados
- coeficientes / odds ratios,
- feature importance,
- permutation importance,
- PDP / ICE,
- SHAP global,
- SHAP individual.

### Mensaje central
En ML moderno, **precisión sin comprensión** puede ser riesgosa.  
Y **comprensión sin validación** también puede ser insuficiente.
Lo correcto es equilibrar:
- desempeño,
- robustez,
- trazabilidad,
- interpretabilidad.



# 16. Cierre tipo industria

## Cuándo priorizar interpretabilidad
- scorecards,
- crédito regulado,
- decisiones públicas,
- modelos de negocio con obligación de justificación.

## Cuándo aceptar modelos más complejos
- cuando la mejora predictiva es relevante,
- cuando el costo del error es alto,
- cuando existen herramientas de gobernanza y explicabilidad,
- cuando el negocio tolera mayor complejidad operativa.

## Regla práctica para tus alumnos
1. empieza con un baseline interpretable,  
2. compara contra modelos más potentes,  
3. mide la mejora real,  
4. aplica explicabilidad,  
5. decide con criterio de negocio.



# 17. Ejercicios propuestos para alumnos

1. Comparar la logística contra Random Forest y explicar cuál elegirían para una entidad financiera.  
2. Revisar si la variable `sex` domina demasiado el modelo y discutir sesgo / fairness.  
3. Cambiar hiperparámetros de XGBoost o LightGBM y comparar:
   - AUC
   - permutation importance
   - SHAP global  
4. Tomar 3 pasajeros del test y redactar una explicación de negocio de cada predicción.  
5. Construir un mini informe ejecutivo:
   - mejor modelo,
   - variables más relevantes,
   - método de explicación más útil,
   - riesgos del modelo.



# 18. Frase de cierre para la clase

> Un modelo interpretable se entiende por diseño.  
> Un modelo explicable necesita herramientas para ser comprendido.  
> En ciencia de datos aplicada, no basta con predecir bien: también hay que poder defender la decisión.
